# Task 7.14 — Locked Benchmark Phase 7 (In-Dist 82k + CFE + 6 Biến thể Suy biến)

> **Mục tiêu**: Chấm một lượt duy nhất, khóa sổ toàn bộ số liệu bảo vệ luận văn và nghiệm thu cổng **G7.3**:
> 1. **In-Distribution 18 tập test** (82.003 ảnh: 8 họ GANs + 10 họ Diffusion) kiểm tra không quên miền cũ (Real ACC >= 90%, Macro ACC >= 90%).
> 2. **Out-of-Distribution CFE** (1.600 ảnh held-out phân tầng) ở cả 2 chế độ Clean và JPEG Q30.
> 3. **Độ bền trước 6 biến thể suy biến vật lý** trên CFE: `q75`, `q50`, `q30`, `webp40`, `blur15`, `downup05`.
> 4. **Phân tích Per-Generator**: Ma trận 5 subset x 4 nhóm bộ sinh.
> 5. **Tổng hợp báo cáo**: Xuất `final_phase7_report.json` (schema v3), `final_phase7_report.md`, `per_generator_phase7.csv` và `roc_phase7.png`.

> **Môi trường**: Google Colab GPU (Tesla T4 / L4 / A100 SXM4) + Google Drive 5TB.

In [ ]:
# CELL 1: KẾT NỐI GOOGLE DRIVE & ĐỒNG BỘ MÃ NGUỒN FATFORMER-XLA (ORIGIN/HOANG)
import os, sys, shutil, subprocess
from pathlib import Path
from google.colab import drive

print("[*] Đang kết nối Google Drive 5TB...")
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/Fatformer')
if not DRIVE_ROOT.exists():
    candidates = [Path('/content/drive/MyDrive/FatFormer_Hub'), Path('/content/drive/MyDrive')]
    for c in candidates:
        if c.exists():
            DRIVE_ROOT = c
            break

print(f"[✓] Thư mục Drive chính: {DRIVE_ROOT}")

REPO_DIR = Path('/content/fatformer-xla')
if not (REPO_DIR / '.git').exists():
    print("[*] Clone nhánh hoang từ GitHub...")
    subprocess.run(['git', 'clone', '-b', 'hoang', 'https://github.com/LeeVHoangtk3/fatformer-xla.git', str(REPO_DIR)], check=True)
else:
    print("[*] Cập nhật nhánh hoang mới nhất...")
    subprocess.run(['git', 'fetch', 'origin', 'hoang'], cwd=REPO_DIR, check=True)
    subprocess.run(['git', 'checkout', 'origin/hoang'], cwd=REPO_DIR, check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))

commit_sha = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
print(f"[✓] Commit hiện tại: {commit_sha}")


In [ ]:
# CELL 2: CÀI ĐẶT THƯ VIỆN & KIỂM TRA PHẦN CỨNG GPU
import torch

print("[*] Đang cài đặt các thư viện cần thiết...")
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ftfy', 'regex', 'tqdm', 'pyyaml', 'timm', 'einops', 'scikit-learn', 'pandas', 'matplotlib', 'Pillow'], check=True)

print("=" * 70)
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"[✓] GPU khả dụng: {gpu_name} ({vram_gb:.2f} GiB VRAM)")
else:
    print("[!] CẢNH BÁO: Không tìm thấy GPU! Đang chạy trên CPU (sẽ rất chậm đối với 82k ảnh).")
print("=" * 70)


In [ ]:
# CELL 3: ĐỊNH VỊ CHECKPOINT, BACKBONE VÀ BUNG DỮ LIỆU TỪ DRIVE SANG SSD COLAB
import tarfile, glob

DATA_STAGE = Path('/content/dataset_local')
DATA_STAGE.mkdir(parents=True, exist_ok=True)

def find_file(candidates):
    for c in candidates:
        if not c:
            continue
        matches = glob.glob(str(c), recursive=True)
        if matches:
            return Path(matches[0])
    return None

print("[*] Đang định vị checkpoint và trọng số CLIP ViT-L/14...")
ckpt_candidates = [
    DRIVE_ROOT / 'checkpoint/phase7/anchored_v2_run1/fatformer_anchored_v2_best.pth',
    DRIVE_ROOT / 'checkpoint/phase7/ablation_v2_fixed_run1/a4_full/fatformer_anchored_v2_best.pth',
    DRIVE_ROOT / 'checkpoint/phase7/ablation_v2_fixed_run1/a4_full/best.pth',
    DRIVE_ROOT / 'checkpoint/phase7/ablation_v2_fixed_run1/a3_data_featcons/best.pth',
    DRIVE_ROOT / '**/fatformer_anchored_v2_best.pth',
    DRIVE_ROOT / '**/fatformer_crossdomain_weights_only.pth'
]
target_ckpt = find_file(ckpt_candidates)
assert target_ckpt and target_ckpt.exists(), f"[!] Không tìm thấy checkpoint nào trong {ckpt_candidates}!"
print(f"[✓] Checkpoint mục tiêu: {target_ckpt}")

clip_candidates = [
    DRIVE_ROOT / 'pretrained/ViT-L-14.pt',
    DRIVE_ROOT / '**/ViT-L-14.pt',
    REPO_DIR / 'DATASET/pretrained/ViT-L-14.pt'
]
clip_path = find_file(clip_candidates)
assert clip_path and clip_path.exists(), f"[!] Không tìm thấy ViT-L-14.pt!"
print(f"[✓] CLIP Backbone: {clip_path}")

print("\n[*] Đang tìm kiếm các tệp nén dữ liệu kiểm chuẩn trên Drive...")
tar_gans = find_file([DRIVE_ROOT / 'datasets/test_benchmark_gans.tar', DRIVE_ROOT / '**/test_benchmark_gans.tar'])
tar_diff = find_file([DRIVE_ROOT / 'datasets/test_benchmark_diffusion.tar', DRIVE_ROOT / '**/test_benchmark_diffusion.tar'])
tar_cfe = find_file([DRIVE_ROOT / 'datasets/test_communityforensics.tar', DRIVE_ROOT / '**/test_communityforensics.tar'])

def extract_bundle(tar_p, target_dir):
    if not tar_p or not tar_p.exists():
        print(f"[!] Bỏ qua (không tìm thấy): {tar_p}")
        return
    print(f"[*] Đang giải nén {tar_p.name} sang {target_dir}...")
    with tarfile.open(tar_p, 'r') as tf:
        tf.extractall(target_dir)
    print(f"[✓] Giải nén xong: {tar_p.name}")

CLEAN_IN_DIST_DIR = DATA_STAGE / 'test_indist'
CLEAN_IN_DIST_DIR.mkdir(parents=True, exist_ok=True)
extract_bundle(tar_gans, CLEAN_IN_DIST_DIR)
extract_bundle(tar_diff, CLEAN_IN_DIST_DIR)

CFE_DIR = DATA_STAGE / 'cfe'
CFE_DIR.mkdir(parents=True, exist_ok=True)
extract_bundle(tar_cfe, CFE_DIR)

# Tự động phát hiện thư mục gốc chứa các subset
cfe_roots = [d.parent for d in CFE_DIR.rglob('1_commercial') if (d / '0_real').is_dir()]
resolved_cfe_dir = cfe_roots[0] if cfe_roots else CFE_DIR
print(f"[✓] Thư mục CFE: {resolved_cfe_dir}")
print(f"[✓] Thư mục In-Dist: {CLEAN_IN_DIST_DIR}")


In [ ]:
# CELL 4: CHẠY BENCHMARK IN-DISTRIBUTION 18 TẬP (82.003 ẢNH)
print("=" * 88)
print("BẮT ĐẦU ĐÁNH GIÁ 18 TẬP IN-DISTRIBUTION (82.003 ẢNH)...")
print("=" * 88)

cmd_indist = [
    sys.executable, 'tools/eval_locked_benchmark.py',
    '--checkpoint', str(target_ckpt),
    '--clip_path', str(clip_path),
    '--test_root', str(CLEAN_IN_DIST_DIR),
    '--calibration', 'DATASET/logs/phase7/calibration/baseline.json',
    '--output_dir', 'DATASET/logs/phase7',
    '--batch_size', '32',
    '--num_workers', '4',
    '--export_phase7'
]

proc = subprocess.Popen(cmd_indist, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
assert proc.returncode == 0, f"Lỗi khi chạy eval_locked_benchmark.py (exit code {proc.returncode})"
print("\n[✓] HOÀN TẤT ĐO ĐẠC 18 TẬP IN-DIST!")


In [ ]:
# CELL 5: CHẠY BENCHMARK CFE (CLEAN, Q30 VÀ 6 BIẾN THỂ SUY BIẾN VẬT LÝ)
print("=" * 88)
print("BẮT ĐẦU ĐÁNH GIÁ CFE HELD-OUT: CLEAN, Q30 VÀ 6 BIẾN THỂ SUY BIẾN...")
print("=" * 88)

cmd_cfe = [
    sys.executable, 'tools/eval_communityforensics.py',
    '--model_type', 'anchored',
    '--model_name', 'anchored_v2',
    '--checkpoint_path', str(target_ckpt),
    '--clip_path', str(clip_path),
    '--cfe_dir', str(resolved_cfe_dir),
    '--condition', 'all_variants',
    '--calibration', 'DATASET/logs/phase7/calibration/baseline.json',
    '--output_dir', 'DATASET/logs/phase7',
    '--batch_size', '32',
    '--num_workers', '4'
]

proc = subprocess.Popen(cmd_cfe, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
assert proc.returncode == 0, f"Lỗi khi chạy eval_communityforensics.py (exit code {proc.returncode})"

# Đánh giá thêm chế độ Clean cho CFE
cmd_cfe_clean = list(cmd_cfe)
cmd_cfe_clean[cmd_cfe_clean.index('all_variants')] = 'clean'
proc = subprocess.Popen(cmd_cfe_clean, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
assert proc.returncode == 0, f"Lỗi khi chạy eval_communityforensics.py clean (exit code {proc.returncode})"

print("\n[✓] HOÀN TẤT ĐO ĐẠC TOÀN BỘ CÁC BIẾN THỂ CFE!")


In [ ]:
# CELL 6: TỔNG HỢP TOÀN DIỆN, ĐÁNH GIÁ 2 CỔNG G7.2/G7.3 & VẼ BIỂU ĐỒ ROC
print("=" * 88)
print("TỔNG HỢP TOÀN BỘ SẢN PHẨM KHÓA SỔ TASK 7.14...")
print("=" * 88)

cmd_agg = [
    sys.executable, 'tools/aggregate_phase7_benchmark.py',
    '--logs_dir', 'DATASET/logs/phase7',
    '--output_dir', 'DATASET/logs/phase7'
]

proc = subprocess.Popen(cmd_agg, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in proc.stdout:
    print(line, end='', flush=True)
proc.wait()
assert proc.returncode == 0, f"Lỗi khi tổng hợp benchmark (exit code {proc.returncode})"
print("\n[✓] HOÀN TẤT TỔNG HỢP BÁO CÁO VÀ BIỂU ĐỒ ROC!")


In [ ]:
# CELL 7: ĐÓNG GÓI SẢN PHẨM, LƯU VỀ GOOGLE DRIVE & TẠO LINK DOWNLOAD
import zipfile
from google.colab import files

OUT_ZIP_LOCAL = Path('/content/task_7.14_final_benchmark_reports.zip')
DRIVE_FINAL_DIR = DRIVE_ROOT / 'logs/phase7'
DRIVE_FINAL_DIR.mkdir(parents=True, exist_ok=True)

target_files = [
    'DATASET/logs/phase7/final_phase7_report.json',
    'DATASET/logs/phase7/final_phase7_report.md',
    'DATASET/logs/phase7/per_generator_phase7.csv',
    'DATASET/logs/phase7/roc_phase7.png',
    'DATASET/logs/phase7/indist_phase7.json',
    'DATASET/logs/phase7/LOCKED_CONFIG.json'
]

for f in glob.glob('DATASET/logs/phase7/eval_v3_*.json'):
    target_files.append(f)
for f in glob.glob('DATASET/logs/phase7/eval_v3_*.md'):
    target_files.append(f)

print(f"[*] Đang đóng gói {len(target_files)} tệp báo cáo vào {OUT_ZIP_LOCAL.name}...")
with zipfile.ZipFile(OUT_ZIP_LOCAL, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for tf in target_files:
        p = Path(tf)
        if p.exists():
            zf.write(p, arcname=p.name)
            print(f"  + Đã thêm: {p.name}")

# Sao chép sang Drive
drive_zip_path = DRIVE_FINAL_DIR / OUT_ZIP_LOCAL.name
shutil.copy2(OUT_ZIP_LOCAL, drive_zip_path)
print(f"\n[✓] ĐÃ LƯU TRỰC TIẾP VÀO GOOGLE DRIVE: {drive_zip_path}")

print("[*] Đang chuẩn bị tải về máy cá nhân...")
files.download(str(OUT_ZIP_LOCAL))
